# Space-map — Xenium (polyp) registration

One-click registration of Xenium polyp serial sections. Runs affine +
non-rigid (LDDMM), then exports:
- an **Imaris multi-layer TIFF** (a 3D `(Z, C, H, W)` stack), and
- a **3D reconstruction** scatter with the layers stacked along Z.

**Data:** set `DATA` to your local `xenium_polyp.csv.gz` (columns `x`, `y`,
`layer`). `LAYERS` limits how many layers to run (`None` = all).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # this repo's space_map wins

import pandas as pd
from space_map import Slice
from space_map.flow import FlowImport, FlowExport, AutoFlowMultiCenter4

# --- edit this to your local file ---
DATA = 'data/xenium_polyp.csv.gz'
X_COL, Y_COL, LAYER_COL = 'x', 'y', 'layer'
LAYERS = None  # first N layers; None = all
OUT = 'out/xenium_polyp'

## Load per-layer coordinates

In [ ]:
df = pd.read_csv(DATA, usecols=[X_COL, Y_COL, LAYER_COL])
layer_ids = sorted(df[LAYER_COL].unique())
if LAYERS:
    layer_ids = layer_ids[:LAYERS]
layers = [df[df[LAYER_COL] == k][[X_COL, Y_COL]].to_numpy(float) for k in layer_ids]
print(f'{len(layers)} layers {layer_ids}, {sum(len(a) for a in layers):,} cells')

## Register (affine + non-rigid)

In [ ]:
fi = FlowImport(f'{OUT}/_work')
slices = fi.init_multi([pd.DataFrame(a, columns=['x', 'y']) for a in layers],
                       ids=[str(k) for k in layer_ids])
af = AutoFlowMultiCenter4(slices, initJKey=Slice.rawKey, alignMethod='auto')
af.affine(useKey='DF', show=False)
af.ldm_pair(Slice.align1Key, Slice.align2Key, show=False)
print('registration done')

## Export: Imaris multi-layer TIFF + 3D reconstruction

In [ ]:
fe = FlowExport(slices)

# Imaris: one single-channel TIFF per layer (1000x1000) -> out/xenium_polyp/imaris/
# named c_<layer>.tiff  (density only here)
fe.export_imaris_tiff(f'{OUT}/imaris', key=Slice.align2Key, size=1000)

# 3D reconstruction: all layers stacked along Z (~45 deg view)
fe.plot_3d(key=Slice.align2Key, channel='DF',
           save=f'{OUT}/reconstruction_3d.png',
           title='Xenium polyp - aligned 3D')
print('wrote', f'{OUT}/imaris/', 'and', f'{OUT}/reconstruction_3d.png')

## Inspect the 3D plot inline

In [ ]:
from IPython.display import Image
Image(filename=f'{OUT}/reconstruction_3d.png')

## Command-line equivalent

The same flow can be run without Jupyter (from the `examples/` directory):

```bash
python run_all.py xenium --data data/xenium_polyp.csv.gz -o out
# or, on your own data, via the space-map CLI:
space-map register data/xenium_polyp.csv.gz --output out/xenium \
    --x-col x --y-col y --layer-col layer
```

Add `--method sift_vgg` for a lighter, checkpoint-free matcher.